# A - Cumulative subducted carbon

Produce grids of cumulative subducted from each contribution of:

- Sediment
- Serpentinite
- Crust
- Lithosphere

through time.

The previous notebook __Subducted Carbon.ipynb__ must be run before this notebook. 

### Note: To save the outputs in this notebook once, the chosen hard drive requires at least 150 GB of free storage.

In [ ]:
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
import joblib
import numpy as np
import pygplates
import os, glob
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
import gplately
import gplately.grids as grids
import gplately.tools as tools
import gplately.ptt as ptt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader as shpreader
import netCDF4
from scipy import ndimage
import pandas as pd
from slabdip import SlabDipper
import numpy.ma as ma
%matplotlib inline
# plt.style.use('ggplot')

#from pygplates_helper import *

# common variables
extent_globe = [-180, 180, -90, 90]
earth_radius = 6371.0e3
earth_surface_area = 4.0*np.pi*earth_radius**2
tessellation_threshold_radians = np.radians(0.01)

# output grid resolution - should be identical to input grid resolution!
spacingX, spacingY = 0.2, 0.2
resX, resY = int(360./0.2 + 1), int(180./0.2 + 1)
lon_grid = np.arange(extent_globe[0], extent_globe[1]+spacingX, spacingX)
lat_grid = np.arange(extent_globe[2], extent_globe[3]+spacingY, spacingY)
lonq, latq = np.meshgrid(lon_grid,lat_grid)

# reconstruction time steps and spacing
min_time = 0
max_time = 170
timestep_size = 1

# time array
# reversed (start at max_time, end at min_time)
reconstruction_times = np.arange(max_time, min_time-timestep_size, -timestep_size)




# OUTPUT SAVING TOGGLES
save_output_netcdf = True # !! important
save_output_snapshots = False

# This is to save outputs from the parallelised "carbon_subducted_and_accreted" routine
save_outputs_to_csv = True

os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
plt.rcParams['font.family'] = 'Helvetica'

### Ensure all needed output paths are created
Change `folder_name` to keep track of results of Notebook 03 reruns

In [ ]:
save_cumulative_subducted_carbon = True
save_smoothed_timestep_grid = True
save_smoothed_cumulative_grid = True

# Save the sum of crust + lithosphere + serpentinite
save_mantle_cumulative_subducted_carbon = True

In [ ]:
# --------------- Everything below here can be kept as-is ----------------
# FOR CUMULATIVE SUBDUCTED CARBON PER TIMESTEP
output_cumulative_subducted_carbon_grid_directory = "../Grids/cumulative_subducted_carbon/{}/{}/"

for storage in ['Lithosphere', 'Serpentinite', 'Crust', 'Sediment', 'Mantle', 'Organic_Sediments']:
    for quantity in ['min', 'mean', 'max']:
        os.makedirs(output_cumulative_subducted_carbon_grid_directory.format(storage, quantity), exist_ok=True)

Define gplately's `PlateReconstruction` and `PlotTopologies` objects.

In [ ]:
model_dir = "./Alfonso_etal_2024_modClennettMuller/"

feature_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/"
        r"*.gpml",
    )
)

rotation_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/",
        r"*.rot",
    )
)
coastlines_filename = os.path.join(
    model_dir,
    "Coastlines",
    "Clennett__etal_2020_Coastlines.gpml",
)

static_polygons = os.path.join(
    model_dir,
    "StaticPolygons/Clennett_2020_StaticPolygons.gpml"
)

model = gplately.PlateReconstruction(
    rotation_model=rotation_filenames,
    topology_features=pygplates.FeatureCollection(
        [
            i for i in pygplates.FeaturesFunctionArgument(
                feature_filenames
            ).get_features()
            if i.get_feature_type().to_qualified_string()
            != "gpml:TopologicalSlabBoundary"
        ]
        
    ),
    static_polygons=static_polygons
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=model,
    continents=coastlines_filename,
)

max_time = 170
min_time = 0
reconstruction_times = np.arange(max_time, min_time-1, -1)

In [ ]:
cell_area = tools.lat_area_function(lat_grid, lat_grid+spacingY, lon_grid.size)
cell_area = np.tile(cell_area.reshape(-1,1), lon_grid.size)

## Make rasters of cumulative subducted carbon

### Build the `cumulative_subducted_carbon` array
If `save_output_netcdf` was set to `True` in Notebook `02-Subducted-Carbon.ipynb`, masked arrays of total global subducted carbon per timestep would have been saved to a netCDF4 grid. All grid nodes with no subducted carbon are given NaN entries.

In the cells below, we loop through time to collect the carbon subducted for each carbon storage reservoir and quantity. At each time step, we append the current contribution to an array, `cumulative_subducted_carbon` for a given component and quantity. Every timestep we save the current cumulative subducted carbon grid to a netcdf file.

To make this process faster, it is performed using parallelisation with `joblib`. The process is made more memory-friendly by partitioning the time array (and thus the parallel tasks) into groups of times. 

In [ ]:
# Define carbon quantities and components
carbon_components = ["Sediment", "Serpentinite", "Crust", "Lithosphere", "Organic_Sediments"]
quantities = ["min", "mean", "max"]

# Define path to the subducted carbon contributions per timestep
subducted_carbon_grid_filename = "../Grids/subducted_carbon/{}/{}/subducted_carbon_{}_{}.nc"

### Function to smooth subducted carbon contribution grids with `scipy.ndimage`'s `fourier_gaussian()` filter
...which we access from an outsourced utility script `fft_gaussian_filter.py` in the `/utils/` directory.

In [ ]:
if save_smoothed_timestep_grid:
    
    import sys
    sys.path.insert(0, './utils/')

    import fft_gaussian_filter as fft_gaussian_filter_script

    def gaussian_smooth_grid(data, distance_km):
   
        """ Smooth grids using a Gaussian. 
        If input grid comes with a mask, all masked values are retained. 
        """
        mask = np.isnan(data)
        smoothed_grid = fft_gaussian_filter_script.fft_gaussian_filter(data, distance_km)
        smoothed_grid[mask] = np.nan
        return smoothed_grid

Select a Gaussian filter width (km) and produce a save directory for all cumulative grids with this smoothing filter width.

In [ ]:
distance_km = 300

# Define path to the cumulative subducted carbon grid we will make in this cell
output_cumulative_subducted_carbon_filename = "../Grids/cumulative_subducted_carbon/{}/{}/cumulative_subducted_carbon_{}_{}.nc"
output_smooth_cumulative_subducted_carbon_grid_directory = "../Grids/smoothed_cumulative_subducted_carbon/{}km/{}/{}/"
output_smooth_cumulative_subducted_carbon_filename = output_smooth_cumulative_subducted_carbon_grid_directory+"cumulative_subducted_carbon_{}_{}.nc"

for storage in ['Mantle', 'Sediment', 'Crust', "Organic_Sediments"]:
    for quantity in ['min', 'mean', 'max']:
        os.makedirs(output_smooth_cumulative_subducted_carbon_grid_directory.format(distance_km, storage, quantity), exist_ok=True)
        
        
output_smoothed_subducted_carbon_grid_directory = "../Grids/smoothed_subducted_carbon/{}km/{}/{}/"
smoothed_subducted_carbon_grid_filename = output_smoothed_subducted_carbon_grid_directory+"subducted_carbon_{}_{}.nc"
for storage in ['Lithosphere', 'Serpentinite', 'Crust', 'Sediment', 'Organic_Sediments']:
    for quantity in ['min', 'mean', 'max']:
        os.makedirs(output_smoothed_subducted_carbon_grid_directory.format(distance_km, storage, quantity), exist_ok=True)

### Accumulate grids in parallel, save to netCDF in a single core 

In [ ]:
def accumulate_subducted_carbon(component, reconstruction_time, distance_km=300):
    
    # Store cumulative subducted carbon from the grids in a numpy array
    subducted_carbon_components = np.zeros((3, resY, resX))
    smoothed_subducted_carbon_components = np.zeros((3, resY, resX))

    for i, quantity in enumerate(quantities):

        subducted_carbon_filename = subducted_carbon_grid_filename.format(
             component, quantity, component.lower(), reconstruction_time
        )

        # Read the masked subducted carbon contribution array for the current time
        current_time_grid = grids.read_netcdf_grid(subducted_carbon_filename)

        # Get the data array, and fill all NaNs with 0s.
        current_time_grid = np.nan_to_num(current_time_grid.data)
        
        current_time_grid_filled = grids.fill_raster(current_time_grid)
        subducted_carbon_components[i] = current_time_grid_filled

        # If the user wants the smoothed piecewise grids to be returned,
        if save_smoothed_timestep_grid:

            
            # Define the filename for saving the smoothed grids
            smoothed_filename = smoothed_subducted_carbon_grid_filename.format(
                     distance_km, component, quantity, component.lower(), reconstruction_time
            )
            # Smooth the grids
            smoothed_current_time_grid = gaussian_smooth_grid(
                current_time_grid_filled,
                
                distance_km=distance_km
            )
            # Read in the smoothed grid, fill the raster, and return
            smoothed_current_time_grid_filled = grids.fill_raster(smoothed_current_time_grid)
            smoothed_subducted_carbon_components[i] += smoothed_current_time_grid_filled
            
            # Create a mask for the interpolated grid - all values with 0s need to have an equivalent
            # mask value of NaN. 
            smoothed_current_time_grid_filled_masked = ma.masked_array(smoothed_current_time_grid_filled)
            smoothed_current_time_grid_filled_masked = ma.masked_values(smoothed_current_time_grid_filled_masked, 0.0)
            smoothed_current_time_grid_filled_masked.fill_value = np.nan
            
            current_subducted_carbon_smoothed = np.ma.array(
                    smoothed_subducted_carbon_components[i],
                    mask = smoothed_current_time_grid_filled_masked.mask,
                    fill_value=np.nan
                )
            
            grids.write_netcdf_grid(
                smoothed_filename,
                current_subducted_carbon_smoothed
            )

    if save_smoothed_timestep_grid:
        return subducted_carbon_components, reconstruction_time, smoothed_current_time_grid_filled, smoothed_filename
    else:
        return subducted_carbon_components, reconstruction_time

        
def accumulate_smoothed_mantle_grids(reconstruction_time, distance_km=300):
    
    # Store cumulative subducted carbon from the grids in a numpy array
    smoothed_subducted_carbon_components = np.zeros((3, resY, resX))

    for i, quantity in enumerate(quantities):
        for c, component in enumerate(["Lithosphere", "Crust", "Serpentinite"]):

            subducted_carbon_filename = smoothed_subducted_carbon_grid_filename.format(
                 distance_km, component, quantity, component.lower(), reconstruction_time
            )

            current_time_grid = grids.read_netcdf_grid(subducted_carbon_filename)
            # Get the data array, and fill all NaNs with 0s.
            current_time_grid = np.nan_to_num(current_time_grid.data)
            current_time_grid_filled = grids.fill_raster(current_time_grid)

            # For the current time, we have a 3 by resY by resX grid. Each dimension in
            # axis 0 is the sum of all crust, serpentinite and lithosphere in the 
            # min, mean and max (respectively)
            smoothed_subducted_carbon_components[i] += current_time_grid_filled

    return smoothed_subducted_carbon_components


def accumulate_reservoir_grids(reconstruction_time, component, distance_km=300):
    
    # Store cumulative subducted carbon from the grids in a numpy array
    smoothed_subducted_carbon_components = np.zeros((3, resY, resX))

    for i, quantity in enumerate(quantities):

        subducted_carbon_filename = smoothed_subducted_carbon_grid_filename.format(
             distance_km, component, quantity, component.lower(), reconstruction_time
        )

        current_time_grid = grids.read_netcdf_grid(subducted_carbon_filename)
        # Get the data array, and fill all NaNs with 0s.
        current_time_grid = np.nan_to_num(current_time_grid.data)
        current_time_grid_filled = grids.fill_raster(current_time_grid)

        # For the current time, we have a 3 by resY by resX grid. Each dimension in
        # axis 0 is the current reservoir subducted carbon grid in min, mean and max (respectively)
        smoothed_subducted_carbon_components[i] += current_time_grid_filled

    return smoothed_subducted_carbon_components

## Accumulate grids in parallel...

This cell can get memory-intensive. We can relieve memory pressure by setting `time_splitter_integer` to a smaller value. It can be 0, or negative. The smaller the value, the smaller the chunks of timesteps used for accumulation.

In [ ]:
time_splitter_integer = -5

In [ ]:
with Parallel(n_jobs=-3, verbose=1) as parallel:
    cumulative_subducted_carbon = np.zeros((len(carbon_components), 3, resY, resX))
    smoothed_cumulative_subducted_carbon = np.zeros((len(carbon_components), 3, resY, resX))
    total_smoothed_cumulative_mantle = np.zeros((3, resY, resX))
    
    # Split the time array into groups of timesteps
    split_times = np.array_split(reconstruction_times, cpu_count() - time_splitter_integer)
    
    # Start counting current time from min_time
    curr_time = min_time
    
    # Loop through each time array. get a group of subducted carbon contributions
    for times in split_times:
        for c, component in enumerate(carbon_components):
            
            if component == "Sediment" and times[-1] > 170.:
                continue
                
            if component == "Organic_Sediments" and times[-1] > 540.:
                continue
            
            print("Accumulating {} grids from {} to {}Ma...".format(component, times[0], times[-1]))
            
            # Accumulate raw carbon grids, and simultaneously smooth the individual contributions.
            results = parallel(delayed(accumulate_subducted_carbon)(
                component, 
                time,
                distance_km=distance_km
            ) 
               for time in times
            )

            results = np.array(results, dtype=object)

            carbon_subducted = results[:,0]
            curr_times = results[:,1]
            
            if save_smoothed_timestep_grid:
                print("Smoothed {} grids from {} to {}Ma, saving grids now...".format(component, times[0], times[-1]))
                smoothed_carbon_subducted = results[:,2]

            # For each time index and result in the current result array,
            for t, subducted_carbon_components in enumerate(carbon_subducted):
                
                # We have one array per timestep with 3 entries for min, mean and max. 
                # print(component, t, len(subducted_carbon_components)) 
                if save_cumulative_subducted_carbon:
                    cumulative_subducted_carbon[c] += subducted_carbon_components 
                
                # Repeat for the smoothed subducted carbon piecewise grids if saving smoothed grids. 
                if save_smoothed_cumulative_grid:
                    smoothed_cumulative_subducted_carbon[c] += smoothed_carbon_subducted[t]
                    
                
                for q, quantity in enumerate(quantities):
                    
                    # Save the raw cumulative subducted carbon grid. 
                    if save_cumulative_subducted_carbon:
                        
                        # Make the mask
                        cumulative_subducted_carbon_curr = ma.masked_array(cumulative_subducted_carbon[c,q])
                        cumulative_subducted_carbon_curr = ma.masked_values(cumulative_subducted_carbon_curr, 0.0)
                        cumulative_subducted_carbon_curr.fill_value = np.nan

                        current_cumulative_subducted_carbon = np.ma.array(
                                cumulative_subducted_carbon[c,q],
                                mask = cumulative_subducted_carbon_curr.mask,
                                fill_value=np.nan
                            )
                        
                        gplately.grids.write_netcdf_grid(
                            output_cumulative_subducted_carbon_filename.format(
                                 
                                component, 
                                quantity, 
                                component.lower(), 
                                curr_times[t]
                            ), 
                            current_cumulative_subducted_carbon
                        )
                        
                    # Save the smoothed cumulative subducted carbon grid if sediment.
                    if save_mantle_cumulative_subducted_carbon: 
                        if component == "Sediment" or component == "Organic_Sediments":
                            if save_smoothed_cumulative_grid: 
                                
                                # Make the mask
                                cumulative_sediment = ma.masked_array(smoothed_cumulative_subducted_carbon[c,q])
                                cumulative_sediment = ma.masked_values(cumulative_sediment, 0.0)
                                cumulative_sediment.fill_value = np.nan
        
                                current_cumulative_subducted_sediment = np.ma.array(
                                        smoothed_cumulative_subducted_carbon[c,q],
                                        mask = cumulative_sediment.mask,
                                        fill_value=np.nan
                                    )
                            
                                gplately.grids.write_netcdf_grid(
                                    output_smooth_cumulative_subducted_carbon_filename.format(
                                         
                                        distance_km,
                                        component, 
                                        quantity, 
                                        component.lower(), 
                                        curr_times[t]
                                    ), 
                                    current_cumulative_subducted_sediment
                                )
                    else:
                        if save_smoothed_cumulative_grid: 
                                
                                # Make the mask
                                cumulative_sediment = ma.masked_array(smoothed_cumulative_subducted_carbon[c,q])
                                cumulative_sediment = ma.masked_values(cumulative_sediment, 0.0)
                                cumulative_sediment.fill_value = np.nan
        
                                current_cumulative_subducted_sediment = np.ma.array(
                                        smoothed_cumulative_subducted_carbon[c,q],
                                        mask = cumulative_sediment.mask,
                                        fill_value=np.nan
                                    )
                            
                                gplately.grids.write_netcdf_grid(
                                    output_smooth_cumulative_subducted_carbon_filename.format(
                                         
                                        distance_km,
                                        component, 
                                        quantity, 
                                        component.lower(), 
                                        curr_times[t]
                                    ), 
                                    current_cumulative_subducted_sediment
                                )
                            
        if save_mantle_cumulative_subducted_carbon:

            mantle_results = parallel(delayed(accumulate_smoothed_mantle_grids)(time, distance_km) for time in times)
            mantle_results_arr = np.array(mantle_results, dtype=object)
            mantle_cumulative_results = mantle_results_arr[:,0]
            
            print("Saving mantle grids (lithosphere + crust + serpentinite) from {} to {}Ma...".format(times[0], times[-1]))
            
            for t, curr_time_mantle_carbon in enumerate(mantle_cumulative_results):
                
                curr_time_mantle_carbon = curr_time_mantle_carbon.astype(dtype=float)
                total_smoothed_cumulative_mantle += curr_time_mantle_carbon
                
                for q, quantity in enumerate(quantities):
                    
                    current_mantle = total_smoothed_cumulative_mantle[q,:]
                    current_mantle_filled = grids.fill_raster(current_mantle)

                    # Save the smoothed cumulative subducted carbon grid. 
                    gplately.grids.write_netcdf_grid(
                        output_smooth_cumulative_subducted_carbon_filename.format(
                             
                            distance_km,
                            "Mantle", 
                            quantity, 
                            "Mantle".lower(), 
                            times[t]
                        ), 
                        current_mantle_filled
                    )
                    
        
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"                    